In [ ]:
from cs336_basics.data import get_random_batch
from cs336_basics.model import BasicsTransformerLM, TransformerBlock, RotaryEmbedding

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-5,
        device=None,
    ):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size, device=device))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        x = x * rms

        return self.weight * x

def pack_hook(t: torch.Tensor):
    addr, shape, dtype, grad_fn = id(t), t.shape, t.dtype, t.grad_fn
    print(f"Saving residual: {addr=}, {shape=}, {dtype=}, {grad_fn=}")
    return t

def unpack_hook(t: torch.Tensor):
    addr, shape, dtype, grad_fn = id(t), t.shape, t.dtype, t.grad_fn
    print(f"Loading residual: {addr=}, {shape=}, {dtype=}, {grad_fn=}")
    return t

x = torch.randn((4, 512, 2560), requires_grad=True)
ln = RMSNorm(x.shape[-1])

with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = ln(x)
    y.sum().backward()

In [ ]:
ln = torch.compile(RMSNorm(x.shape[-1]))

with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    y = ln(x)
    y.sum().backward()

In [ ]:
from typing import Callable

def inspect_function(fn: Callable[[torch.Tensor], torch.Tensor], x: torch.Tensor, backward: bool = False, debug: bool = False):
    # Now logs the number of bytes saved
    total_size_bytes = 0

    def pack_hook(t):
        if isinstance(t, torch.nn.Parameter): # Skip logging parameters to avoid double counting
            return t
        nonlocal total_size_bytes
        shape, dtype, grad_fn = t.shape, t.dtype, t.grad_fn
        total_size_bytes += t.numel() * t.element_size()
        if debug:
            print(f"Saving residual: {shape=}, {dtype=}, {grad_fn=}")
        return t

    def unpack_hook(t: torch.Tensor):
        addr, shape, dtype, grad_fn = id(t), t.shape, t.dtype, t.grad_fn
        if debug:
            print(f"Loading residual: {addr=}, {shape=}, {dtype=}, {grad_fn=}")
        return t

    with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
        x = fn(x)

        if backward:
            if debug:
                print("=== Backward ===")
            x.sum().backward()

    print(f"Total size of saved tensors: {total_size_bytes / (1024**2):.2f} MiB")
    return total_size_bytes

In [ ]:
d_model, d_ff, num_heads, context_length = 2560, 10240, 16, 2048

pos_encoder = RotaryEmbedding(context_length, d_model // num_heads)
block = TransformerBlock(d_model, num_heads, d_ff, pos_encoder)

block = torch.compile(block, fullgraph=True)
x = torch.randn((4, context_length, d_model), requires_grad=True)

def four_block(x):
    for _ in range(4):
        x = block(x)
    return x

# inspect_function(four_block, x)

In [ ]:
from torch.utils.checkpoint import checkpoint

def two_blocks(x):
    x = block(x)
    x = block(x)
    return x

def four_blocks_checkpoint(x):
    # checkpoint throws out all the saved tensors until the backward pass
    # when getting to the checkpointed block in the backward pass,
    # it reruns a forward pass to produce the saved tensors,
    # then completes normal backward pass.
    x = checkpoint(two_blocks, x, use_reentrant=False)
    x = checkpoint(two_blocks, x, use_reentrant=False)
    return x

inspect_function(four_blocks_checkpoint, x, backward=True)

In [ ]:
from cs336_systems.gradient_checkpoint import recursive_checkpoint, linear_checkpoint

inspect_function(lambda x: linear_checkpoint(block, x, num_layers=4, group_size=2), x)

In [ ]:

inspect_function(lambda x: recursive_checkpoint(block, x, num_layers=16), x)

In [ ]:
for group_size in range(1, 17):
    print(f"Group size: {group_size}")
    s = inspect_function(lambda x: linear_checkpoint(block, x, num_layers=16, group_size=group_size), x)